In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Labs"
)

xpt_file = DATA_DIR / "FASTQX_I.xpt"
csv_file = DATA_DIR / "FASTQX_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 FASTQX_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PHQ020",
    "PHACOFHR",
    "PHACOFMN",
    "PHQ030",
    "PHAALCHR",
    "PHAALCMN",
    "PHQ040",
    "PHAGUMHR",
    "PHAGUMMN",
    "PHQ050",
    "PHAANTHR",
    "PHAANTMN",
    "PHQ060",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "FASTQX_I columns/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9165:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 19:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,165 rows x 19 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached FASTQX_I:
#
#     genuine decimal variables = NONE
#     string variables          = NONE
#
# All populated values are whole-number values.
#
#
# Therefore all 19 variables may be converted to
# nullable Int64 after the validated tiny-zero repair.
#
#
# IMPORTANT:
#
# This applies ONLY to FASTQX_I.
#
# It does NOT change your broader fidelity rule:
#
#     Dietary decimal values      -> AS IS
#     Laboratory decimal values   -> AS IS
#     Examination measurements    -> AS IS
#     Survey/sample weights       -> AS IS
#
# True continuous decimal variables are NEVER rounded
# merely to remove ".0".
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 19:

    raise ValueError(
        "Expected exactly 19 whole-number/code variables."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# NEVER globally replace this before validating the
# exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 19 FASTQX_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached FASTQX_I.xpt:
#
# PHDSESN     4,166
# PHAFSTHR    1,048
# PHAFSTMN      171
# PHAGUMHR      122
# PHACOFHR       14
# PHASUPHR       11
# PHAGUMMN        8
# PHAALCHR        6
# PHACOFMN        4
# PHAANTHR        3
# PHASUPMN        2
# PHAANTMN        1
#
# TOTAL        5,556
#
#
# CDC documentation permits zero for these variables:
#
#     PHACOFHR   0-10
#     PHACOFMN   0-58
#
#     PHAALCHR   0-15
#
#     PHAGUMHR   0-14
#     PHAGUMMN   0-59
#
#     PHAANTHR   0-14
#     PHAANTMN   0-59
#
#     PHASUPHR   0-16
#     PHASUPMN   0-59
#
#     PHAFSTHR   0-37
#     PHAFSTMN   0-59
#
# PHDSESN:
#
#     0 = Morning
#     1 = Afternoon
#     2 = Evening
#

EXPECTED_TINY_COUNTS = {
    "PHACOFHR": 14,
    "PHACOFMN": 4,
    "PHAALCHR": 6,
    "PHAGUMHR": 122,
    "PHAGUMMN": 8,
    "PHAANTHR": 3,
    "PHAANTMN": 1,
    "PHASUPHR": 11,
    "PHASUPMN": 2,
    "PHAFSTHR": 1048,
    "PHAFSTMN": 171,
    "PHDSESN": 4166
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated FASTQX_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 5556:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 12:

    raise ValueError(
        "Expected tiny values in exactly 12 variables."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before tiny-value correction."
    )


print(
    "\nPASS: Exact 5,556-value tiny-zero pattern "
    "across 12 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Only restore zero in variables whose complete
# tiny-value pattern has already been validated.
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 5556:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents the exact corrected XPT.
#
# The ONLY permitted numerical modification is:
#
#     validated tiny value -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY as a diagnostic comparison.
#
# No data are rounded or rewritten.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimals found "
        "in FASTQX_I.\n\n"
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "\nPASS: No genuine fractional variables "
    "in FASTQX_I."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 12. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# All 19 fields are whole-number variables.
#
# NO .round() is used during actual conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 19 FASTQX_I fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 13. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 14. HELPER - RANGE CHECK
# ============================================================

def check_range(
    column,
    low,
    high,
    expected_nonmissing,
    expected_missing,
    expected_zero=None
):

    s = df[column]


    actual_nonmissing = int(
        s.notna()
        .sum()
    )


    actual_missing = int(
        s.isna()
        .sum()
    )


    actual_range_count = int(
        s.between(
            low,
            high
        )
        .sum()
    )


    actual_zero = int(
        s.eq(0)
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        f"Range {low}-{high}:",
        f"{actual_range_count:,}"
    )

    print(
        "Nonmissing:",
        f"{actual_nonmissing:,}"
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )

    print(
        "Zero:",
        f"{actual_zero:,}"
    )


    if actual_nonmissing != expected_nonmissing:

        raise ValueError(
            f"{column}: nonmissing count differs."
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column}: missing count differs."
        )


    if actual_range_count != expected_nonmissing:

        raise ValueError(
            f"{column}: values outside CDC range."
        )


    if expected_zero is not None:

        if actual_zero != expected_zero:

            raise ValueError(
                f"{column}: zero count differs.\n"
                f"Expected: {expected_zero:,}\n"
                f"Actual:   {actual_zero:,}"
            )


    print(
        f"PASS: {column} range/count validated."
    )


# ============================================================
# 15. CDC CHECK - PHQ020
# ============================================================

#
# Coffee or tea with cream or sugar?
#
# 1 Yes = 68
# 2 No  = 8,843
# Missing = 254
#

check_frequency(
    "PHQ020",
    {
        1: 68,
        2: 8843
    },
    254
)


# ============================================================
# 16. CDC CHECK - PHACOFHR / PHACOFMN
# ============================================================

check_range(
    "PHACOFHR",
    low=0,
    high=10,
    expected_nonmissing=68,
    expected_missing=9097,
    expected_zero=14
)


check_range(
    "PHACOFMN",
    low=0,
    high=58,
    expected_nonmissing=68,
    expected_missing=9097,
    expected_zero=4
)


# ============================================================
# 17. PHQ020 ROUTING CHECK
# ============================================================

#
# Time variables should be populated exactly when:
#
#     PHQ020 = 1
#

coffee_yes = (
    df["PHQ020"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


coffee_hr_present = (
    df["PHACOFHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


coffee_mn_present = (
    df["PHACOFMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


coffee_hr_mismatch = int(
    np.sum(
        coffee_yes
        !=
        coffee_hr_present
    )
)


coffee_mn_mismatch = int(
    np.sum(
        coffee_yes
        !=
        coffee_mn_present
    )
)


if coffee_hr_mismatch != 0:

    raise ValueError(
        "PHACOFHR routing differs from PHQ020."
    )


if coffee_mn_mismatch != 0:

    raise ValueError(
        "PHACOFMN routing differs from PHQ020."
    )


print(
    "PASS: Coffee/tea fasting-time routing "
    "validated exactly."
)


# ============================================================
# 18. CDC CHECK - PHQ030
# ============================================================

check_frequency(
    "PHQ030",
    {
        1: 38,
        2: 8873
    },
    254
)


# ============================================================
# 19. CDC CHECK - PHAALCHR / PHAALCMN
# ============================================================

check_range(
    "PHAALCHR",
    low=0,
    high=15,
    expected_nonmissing=38,
    expected_missing=9127,
    expected_zero=6
)


#
# CDC range starts at 6 minutes for this specific field.
#
check_range(
    "PHAALCMN",
    low=6,
    high=58,
    expected_nonmissing=38,
    expected_missing=9127,
    expected_zero=0
)


# ============================================================
# 20. PHQ030 ROUTING CHECK
# ============================================================

alcohol_yes = (
    df["PHQ030"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


alcohol_hr_present = (
    df["PHAALCHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


alcohol_mn_present = (
    df["PHAALCMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        alcohol_yes
        !=
        alcohol_hr_present
    )
) != 0:

    raise ValueError(
        "PHAALCHR routing differs from PHQ030."
    )


if int(
    np.sum(
        alcohol_yes
        !=
        alcohol_mn_present
    )
) != 0:

    raise ValueError(
        "PHAALCMN routing differs from PHQ030."
    )


print(
    "PASS: Alcohol fasting-time routing validated."
)


# ============================================================
# 21. CDC CHECK - PHQ040
# ============================================================

check_frequency(
    "PHQ040",
    {
        1: 221,
        2: 8690
    },
    254
)


# ============================================================
# 22. CDC CHECK - PHAGUMHR / PHAGUMMN
# ============================================================

check_range(
    "PHAGUMHR",
    low=0,
    high=14,
    expected_nonmissing=221,
    expected_missing=8944,
    expected_zero=122
)


check_range(
    "PHAGUMMN",
    low=0,
    high=59,
    expected_nonmissing=221,
    expected_missing=8944,
    expected_zero=8
)


# ============================================================
# 23. PHQ040 ROUTING CHECK
# ============================================================

gum_yes = (
    df["PHQ040"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


gum_hr_present = (
    df["PHAGUMHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


gum_mn_present = (
    df["PHAGUMMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        gum_yes
        !=
        gum_hr_present
    )
) != 0:

    raise ValueError(
        "PHAGUMHR routing differs from PHQ040."
    )


if int(
    np.sum(
        gum_yes
        !=
        gum_mn_present
    )
) != 0:

    raise ValueError(
        "PHAGUMMN routing differs from PHQ040."
    )


print(
    "PASS: Gum/mint fasting-time routing validated."
)


# ============================================================
# 24. CDC CHECK - PHQ050
# ============================================================

check_frequency(
    "PHQ050",
    {
        1: 53,
        2: 8858
    },
    254
)


# ============================================================
# 25. CDC CHECK - PHAANTHR / PHAANTMN
# ============================================================

check_range(
    "PHAANTHR",
    low=0,
    high=14,
    expected_nonmissing=53,
    expected_missing=9112,
    expected_zero=3
)


check_range(
    "PHAANTMN",
    low=0,
    high=59,
    expected_nonmissing=53,
    expected_missing=9112,
    expected_zero=1
)


# ============================================================
# 26. PHQ050 ROUTING CHECK
# ============================================================

antacid_yes = (
    df["PHQ050"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


antacid_hr_present = (
    df["PHAANTHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


antacid_mn_present = (
    df["PHAANTMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        antacid_yes
        !=
        antacid_hr_present
    )
) != 0:

    raise ValueError(
        "PHAANTHR routing differs from PHQ050."
    )


if int(
    np.sum(
        antacid_yes
        !=
        antacid_mn_present
    )
) != 0:

    raise ValueError(
        "PHAANTMN routing differs from PHQ050."
    )


print(
    "PASS: Antacid fasting-time routing validated."
)


# ============================================================
# 27. CDC CHECK - PHQ060
# ============================================================

check_frequency(
    "PHQ060",
    {
        1: 123,
        2: 8788
    },
    254
)


# ============================================================
# 28. CDC CHECK - PHASUPHR / PHASUPMN
# ============================================================

check_range(
    "PHASUPHR",
    low=0,
    high=16,
    expected_nonmissing=123,
    expected_missing=9042,
    expected_zero=11
)


check_range(
    "PHASUPMN",
    low=0,
    high=59,
    expected_nonmissing=123,
    expected_missing=9042,
    expected_zero=2
)


# ============================================================
# 29. PHQ060 ROUTING CHECK
# ============================================================

supplement_yes = (
    df["PHQ060"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


supplement_hr_present = (
    df["PHASUPHR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


supplement_mn_present = (
    df["PHASUPMN"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        supplement_yes
        !=
        supplement_hr_present
    )
) != 0:

    raise ValueError(
        "PHASUPHR routing differs from PHQ060."
    )


if int(
    np.sum(
        supplement_yes
        !=
        supplement_mn_present
    )
) != 0:

    raise ValueError(
        "PHASUPMN routing differs from PHQ060."
    )


print(
    "PASS: Supplement fasting-time routing validated."
)


# ============================================================
# 30. CDC CHECK - TOTAL FOOD FAST
# ============================================================

#
# PHAFSTHR:
#
#     0-37 hours
#     n = 8,911
#     missing = 254
#
#
# PHAFSTMN:
#
#     0-59 minutes
#     n = 8,911
#     missing = 254
#

check_range(
    "PHAFSTHR",
    low=0,
    high=37,
    expected_nonmissing=8911,
    expected_missing=254,
    expected_zero=1048
)


check_range(
    "PHAFSTMN",
    low=0,
    high=59,
    expected_nonmissing=8911,
    expected_missing=254,
    expected_zero=171
)


# ============================================================
# 31. TOTAL FAST MISSING-MASK CHECK
# ============================================================

fast_hr_missing = (
    df["PHAFSTHR"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


fast_mn_missing = (
    df["PHAFSTMN"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


fast_mask_mismatch = int(
    np.sum(
        fast_hr_missing
        !=
        fast_mn_missing
    )
)


print(
    "\n--- FASTING HOUR/MINUTE MASK CHECK ---"
)

print(
    "Missing-mask mismatches:",
    fast_mask_mismatch
)


if fast_mask_mismatch != 0:

    raise ValueError(
        "PHAFSTHR and PHAFSTMN missing masks differ."
    )


print(
    "PASS: Fasting hour/minute availability "
    "matches exactly."
)


# ============================================================
# 32. CDC CHECK - PHDSESN
# ============================================================

#
# Session in which participant was examined:
#
#     0 = Morning      4,166
#     1 = Afternoon    3,390
#     2 = Evening      1,609
#
# No missing.
#
#
# This is particularly strong validation that the
# 4,166 tiny PHDSESN values really represent zero.
#

check_frequency(
    "PHDSESN",
    {
        0: 4166,
        1: 3390,
        2: 1609
    },
    0
)


# ============================================================
# 33. FINAL ZERO PATTERN
# ============================================================

#
# Original pandas-decoded XPT contained:
#
#     ordinary zeros = 0
#
# After correction, the zeros must correspond EXACTLY
# to the 5,556 validated tiny-value positions.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from validated "
        "FASTQX_I pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 5556:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 5,556 validated restorations."
)


# ============================================================
# 34. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY permitted numerical changes are:
#
#     validated tiny value -> 0
#
# No other value can change.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 19 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 35. EXPORT CSV
# ============================================================

#
# Standard NHANES fidelity export.
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
#
# FASTQX_I happens to contain no genuine decimal fields,
# but using this standard remains important across your
# other NHANES datasets.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 36. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 37. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 19 variables."
)


# ============================================================
# 38. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 39. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 40. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 41. RAW CSV .0 CHECK
# ============================================================

#
# FASTQX_I has NO genuine decimal fields.
#
# Therefore checking all 19 fields for unnecessary ".0"
# is safe for THIS file.
#
#
# Example:
#
# SEQN:
#     83732.0 -> 83732
#
# PHAFSTHR:
#     12.0 -> 12
#
# PHDSESN:
#     0.0 -> 0
#
#
# DO NOT use this global check on files that contain
# genuine continuous decimals.
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in "
    "any FASTQX_I variable."
)


# ============================================================
# 42. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "the 5,556 validated zero values."
)


# ============================================================
# 43. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 44. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 45. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FASTQX_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Laboratory - Fasting Questionnaire"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables:",
    "NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "Morning examination sessions:",
    int(
        df["PHDSESN"]
        .eq(0)
        .sum()
    )
)


print(
    "Afternoon examination sessions:",
    int(
        df["PHDSESN"]
        .eq(1)
        .sum()
    )
)


print(
    "Evening examination sessions:",
    int(
        df["PHDSESN"]
        .eq(2)
        .sum()
    )
)


print(
    "PHAFSTHR zero values:",
    int(
        df["PHAFSTHR"]
        .eq(0)
        .sum()
    )
)


print(
    "PHAFSTMN zero values:",
    int(
        df["PHAFSTMN"]
        .eq(0)
        .sum()
    )
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "FASTQX_I contains no genuine decimal fields."
)


print(
    "All 19 variables are safely exported as "
    "whole-number values after validated zero restoration."
)


print(
    "This does NOT alter the rule for Dietary or other "
    "continuous decimal variables, which remain exactly AS IS."
)

NHANES 2015-2016 FASTQX_I XPT -> CSV
Rows: 9,165
Columns: 19
PASS: Structure = 9,165 rows x 19 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number/code fields: 19

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 5,556
Variables containing tiny artifacts: 12

Tiny values by variable:
PHDSESN     4166
PHAFSTHR    1048
PHAFSTMN     171
PHAGUMHR     122
PHACOFHR      14
PHASUPHR      11
PHAGUMMN       8
PHAALCHR       6
PHACOFMN       4
PHAANTHR       3
PHASUPMN       2
PHAANTMN       1

PASS: Exact 5,556-value tiny-zero pattern across 12 variables validated.

--- VALIDATED ZERO RESTORATION ---
PHACOFHR: 14 restored to 0
PHACOFMN: 4 restored to 0
PHAALCHR: 6 restored to 0
PHAGUMHR: 122 restored to 0
PHAGUMMN: 8 restored to 0
PHAANTHR: 3 restored to 0
PHAANTMN: 1 restored to 0
PHASUPHR: 11 restored to 0
PHASUPMN: 2 restored to 0
PHAFSTHR: 1,048 restored to 0
PHAFSTMN: 171 restored to 0
PHDSESN: 4,166 r